In [9]:
#loading of the excel file
import pandas as pd

df = pd.read_excel("/content/NL2SQL_10000_Sales_Data.xlsx")
print(df.shape)
print()
print(df.head())


(10000, 8)

   SaleID ProductName     Category       City  Quantity  Amount    SaleDate  \
0       1      Mobile  Electronics    Madurai         1   33024  2025-08-17   
1       2      Mobile  Electronics  Hyderabad         9  186273  2025-02-02   
2       3      Mobile  Electronics    Chennai         4  120988  2026-07-29   
3       4      Webcam  Accessories      Kochi         7   23135  2026-08-27   
4       5      Laptop  Electronics     Trichy         3  218088  2025-10-12   

      Status  
0  Completed  
1  Completed  
2  Completed  
3  Completed  
4  Completed  


In [10]:
#Create Database
import sqlite3

conn = sqlite3.connect("sales.db")
cursor = conn.cursor()

df.to_sql("sales", conn, if_exists = "replace", index = False)
print("Data loaded successfully")

Data loaded successfully


In [42]:
#Retrive Schema
cursor.execute("PRAGMA table_info(sales)")

schema = "Table: sales\nColumns:"
schema_rows = cursor.fetchall()

for column in schema_rows:
  schema += f"{column[1]} {column[2]}\n"

print(schema)


Table: sales
Columns:SaleID INTEGER
ProductName TEXT
Category TEXT
City TEXT
Quantity INTEGER
Amount INTEGER
SaleDate TEXT
Status TEXT



In [15]:
#SQL Query validation
def validate_sql(sql):
  sql_upper = sql.strip().upper()

  if not sql_upper.startswith("SELECT"):
    return False

  dangerous_commands = [ "INSERT", "UPDATE","DELETE",
                        "ALTER","CREATE","REPLACE","ATTACH","DETACH"]
  for command in dangerous_commands:
    if command in sql_upper:
      return False
  return True
print(validate_sql("SELECT * FROM sales LIMIT 5"))
print(validate_sql("DELETE FROM sales"))

True
False


In [23]:
#Execution of SQL after validation
def execute_sql(sql):

  if not validate_sql(sql):
    return "Unsafe SQL Query"

  try:
    cursor.execute(sql)
    result = cursor.fetchall()


    if not result:
      return "No results found"

    return result

  except sqlite3.Error as e:
    return f"SQL Error: {e}"

print(execute_sql("SELECT * FROM sales LIMIT 5"))
print()
print(execute_sql("DELETE FROM sales"))
print()
print(execute_sql("SELECT * FORM sales"))
print()
print(execute_sql("SELECT * FROM sales WHERE city = 'NonExistingCity'"))

[(1, 'Mobile', 'Electronics', 'Madurai', 1, 33024, '2025-08-17', 'Completed'), (2, 'Mobile', 'Electronics', 'Hyderabad', 9, 186273, '2025-02-02', 'Completed'), (3, 'Mobile', 'Electronics', 'Chennai', 4, 120988, '2026-07-29', 'Completed'), (4, 'Webcam', 'Accessories', 'Kochi', 7, 23135, '2026-08-27', 'Completed'), (5, 'Laptop', 'Electronics', 'Trichy', 3, 218088, '2025-10-12', 'Completed')]

Unsafe SQL Query

SQL Error: near "FORM": syntax error

No results found


In [30]:
#Automation template

question = input("Ask your question: ")

if question.lower() == "what is the total sales amount":
  sql = "SELECT SUM(sales) FROM sales"

elif question.lower() == "what is the total quantity sold":
  sql = "SELECT SUM(quantity) FROM sales"

elif question.lower() == "what is the total sales amount for each city":
  sql = "SELECT city, SUM(sales) FROM sales GROUP BY city"

elif question.lower() == "what is the total quantity sold for each product":
  sql = "SELECT product, SUM(quantity) FROM sales GROUP BY product"

else:
  sql = None

if sql:
  result = execute_sql(sql)
  print(result)
else:
  print("Sorry, i dont understand this question.")



Ask your question: what is the total quantity sold
[(54911,)]


In [36]:
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [37]:
from google import genai

client = genai.Client(api_key=api_key)

print("Gemini connected successfully")

Gemini connected successfully


In [52]:
question = "What is the total sales amount?"

prompt = f"""
You are a SQL assitant

Convert the following question into a SQLite SQL query.

Database schema:
{schema}

Question: {question}

Rules:
- Use only the table and coulmns provided in the schema
- Return only the SQL query. Do not explain anything.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

sql = response.text.strip()

print("Generated SQl:")
print(sql)

sql = sql.replace("```sql", "").replace("```", "").strip()

cursor.execute(sql)

result = cursor.fetchone()

print("Query Result:")
print(result)

Generated SQl:
```sql
SELECT SUM(Amount) FROM sales;
```
Query Result:
(1063120064,)


In [55]:
question = input("Ask your question:")

prompt = f"""
You are a SQL assitant

Convert the following question into a SQLite SQL query.

Database schema:
{schema}

Question: {question}

Rules:
- Use only the table and coulmns provided in the schema
- Return only the SQL query. Do not explain anything.
"""

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=prompt
)

sql = response.text.strip()

print("Generated SQl:")
print(sql)

sql = sql.replace("```sql", "").replace("```", "").strip()

cursor.execute(sql)

result = cursor.fetchall()

print("Query Result:")
print(result)

Ask your question:what is the sales per city
Generated SQl:
```sql
SELECT City, SUM(Amount) AS TotalSales FROM sales GROUP BY City;
```
Query Result:
[('Bengaluru', 142465989), ('Chennai', 130700759), ('Coimbatore', 134057045), ('Hyderabad', 131006049), ('Kochi', 125799601), ('Madurai', 140125173), ('Salem', 129826378), ('Trichy', 129139070)]


In [45]:
models = client.models.list()

for model in models:
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/